# MVPy Active Teacher Colab

Runtime: GPU. Prefer L4/A100. Use this notebook to generate teacher data, verify it with MVPy v0.1, then train Qwen3.5/Gemma4 with Unsloth.

In [ ]:
from google.colab import userdata
from google.colab.errors import SecretNotFoundError
import os, pathlib
def secret(name):
    try:
        return userdata.get(name) or ''
    except SecretNotFoundError:
        return ''
os.environ['OPENAI_API_KEY'] = secret('OPENAI_API_KEY')
os.environ['ANTHROPIC_API_KEY'] = secret('ANTHROPIC_API_KEY')
os.environ['HF_TOKEN'] = secret('HF_TOKEN')
WORK = pathlib.Path('/content/mvpy_colab_work')
WORK.mkdir(exist_ok=True)
print('work', WORK)
print('OPENAI_API_KEY', 'set' if os.environ['OPENAI_API_KEY'] else 'missing')
print('HF_TOKEN', 'set' if os.environ['HF_TOKEN'] else 'missing')

In [ ]:
!nvidia-smi
!apt-get update -y >/dev/null
!apt-get install -y golang-go >/dev/null
!pip install -q -U openai anthropic tenacity tqdm datasets pandas

Clone the research branch. If GitHub is unavailable, upload `mvpy_colab_bundle.tar.gz` from local `results/mvpy_colab/`.

In [ ]:
import pathlib, os
REPO = 'https://github.com/vaddisrinivas/voltsnip.git'
BRANCH = 'codex/mvpy-papers-1-3'
if not pathlib.Path('/content/moltsnip').exists():
    !git clone --depth 1 --branch "$BRANCH" "$REPO" /content/moltsnip
os.chdir('/content/moltsnip/vsevals')
print('cwd', os.getcwd())
if not pathlib.Path('data/mvpy_ood_500/ood_qwen.jsonl').exists():
    BUNDLE = '/content/mvpy_colab_bundle.tar.gz'
    if not pathlib.Path(BUNDLE).exists():
        from google.colab import files
        uploaded = files.upload()
        BUNDLE = next(iter(uploaded.keys()))
    !tar -xzf "$BUNDLE" -C /content
    os.chdir('/content/mvpy_colab')
    print('cwd', os.getcwd())
!python colab/mvpy_active_teacher.py env-report

In [ ]:
!python colab/mvpy_active_teacher.py build-verifier \
  --repo https://github.com/vaddisrinivas/mvpy.git \
  --commit 671aecf \
  --out /content/mvpy-v0.1
!/content/mvpy-v0.1 --help || true

Teacher smoke. Use `--oracle` for supervised training rows; remove it for compiler-only repair rows.

In [ ]:
import os, shutil, pathlib
out = pathlib.Path('results/sft/medium_plan_smoke')
if os.environ.get('OPENAI_API_KEY'):
    !python colab/mvpy_active_teacher.py teacher \
      --dataset data/mvpy_ood_500/ood_qwen.jsonl \
      --out results/teacher/medium_oracle_smoke.jsonl \
      --provider openai \
      --model gpt-5.3-codex \
      --band medium \
      --limit 10 \
      --repair-turns 2 \
      --oracle
else:
    print('OPENAI_API_KEY missing; skipping active teacher and using checked-in plan SFT smoke data')
    out.mkdir(parents=True, exist_ok=True)
    shutil.copyfile('data/mvpy_research_mlx_plan_v2/train.jsonl', out / 'train.jsonl')
    shutil.copyfile('data/mvpy_research_mlx_plan_v2/valid.jsonl', out / 'valid.jsonl')

In [ ]:
import pathlib, os
if pathlib.Path('results/teacher/medium_oracle_smoke.jsonl').exists():
    !python colab/mvpy_active_teacher.py prepare-sft \
      --accepted results/teacher/medium_oracle_smoke.jsonl \
      --out-dir results/sft/medium_plan_smoke \
      --target plan_code
else:
    print('using existing results/sft/medium_plan_smoke')
!head -n 1 results/sft/medium_plan_smoke/train.jsonl

Install Unsloth. Restart runtime if Colab asks, then rerun from this cell onward.

In [ ]:
import os, re, torch
!pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
!pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install -q --no-deps --upgrade "torchao>=0.16.0"
!pip install -q --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0" torchcodec timm

Qwen first. Use bf16 LoRA (`--load-in-4bit` off).

In [ ]:
!python colab/train_unsloth_mvpy.py \
  --family qwen35 \
  --model unsloth/Qwen3.5-4B \
  --data-dir results/sft/medium_plan_smoke \
  --out-dir results/train/qwen35_4b_medium_plan_smoke \
  --max-seq-length 1024 \
  --max-steps 20 \
  --rank 16

Gemma second. Use 4-bit on smaller GPUs.

In [ ]:
!python colab/train_unsloth_mvpy.py \
  --family gemma4 \
  --model unsloth/gemma-4-E4B-it \
  --load-in-4bit \
  --data-dir results/sft/medium_plan_smoke \
  --out-dir results/train/gemma4_e4b_medium_plan_smoke \
  --max-seq-length 1024 \
  --max-steps 20 \
  --rank 16

In [ ]:
!tar -czf /content/results_$(date -u +%Y%m%dT%H%M%SZ).tar.gz results
!ls -lh /content/results_*.tar.gz | tail